# 01 · Tokenizer

**Belgedeki bölüm:** Tokenizer

- Tokenize: metni, modelin işleyebileceği küçük parçalara (token) bölmek. Her token'ın sözlükte (vocab) bir ID'si var.
- 3 tür: kelime, harf ve alt kelime (subword) düzeyi. Modern modeller BPE / WordPiece kullanır.
- Detokenize: ID'leri tekrar okunabilir metne çevirmek. Büyük/küçük harf ve boşluk önemlidir.

In [ ]:
# Gerekli paketler (bir kez çalıştırmak yeterli)
%pip install -q tiktoken tokenizers sentencepiece transformers

![https://miro.medium.com/v2/resize:fit:850/1*nhwzi5yDdkI70uhx2KH1Dw.png](https://miro.medium.com/v2/resize:fit:850/1*nhwzi5yDdkI70uhx2KH1Dw.png)

In [ ]:
text1 = "the cat chased the dog"
text2 = "the dog chased the cat"

text = "the capital of france is"

In [ ]:
# basic tokenization
def tokenize(text):
  return text.split()

In [ ]:
tokenize(text1)

In [ ]:
import json

with open("tokenizer.json", "r") as f:
  vocab = json.load(f)

def tokenize2(text):
  parts = text.split()
  ids = []
  for part in parts:
    if part in vocab:
      value = vocab[part]
    else:
      value = vocab["<unk>"]
    ids.append(value)
  return ids

token_ids1 = tokenize2(text1)
token_ids1

In [ ]:
tokenize2("How are you?")

https://arxiv.org/abs/2502.07057

In [ ]:
reverse_vocab = {id: part for part, id in vocab.items()}
reverse_vocab

In [ ]:
def detokenize(ids):
  text = ""
  for id in ids:
    part = reverse_vocab[id]
    text += part + " "
  text = text.strip()
  return text
  
detokenize(token_ids1)

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("gpt2") #Modelin sözlüğünü belleğe yükler

gpt2_ids = enc.encode(text1)
gpt2_ids

In [ ]:
enc.decode(gpt2_ids), enc.decode(token_ids1)

In [ ]:
enc.decode([25])

In [ ]:
enc = tiktoken.get_encoding("o200k_base") #Kelimenin sözlüğünü yükle

enc.n_vocab

In [ ]:
o200_ids = enc.encode(text1)
o200_ids

## İsteğe bağlı: Gemma tokenizer'ı
Gemma modelleri Hugging Face'te izinli (gated) olduğu için önce giriş yapman gerekir. Token'ını koda yazma; `login()` çalışınca açılan kutuya gir.

In [ ]:
from huggingface_hub import login

login()

In [ ]:
# AutoTokenizer gönderdiğin metni modelin anlayabileceği token'lara çevirir.
from transformers import AutoTokenizer

gemma_tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-27b-it")

In [ ]:
gemma_ids = gemma_tokenizer.encode(text1)
gemma_ids

In [ ]:
gemma_tokenizer.vocab_size

In [ ]:
gemma_tokenizer.decode(gemma_ids), gemma_tokenizer.decode(gemma_ids)[5:]

In [ ]:
import json

with open("tokenizer_gemma.json", "w", encoding="utf-8") as f:
  json.dump(gemma_tokenizer.get_vocab(), f, ensure_ascii=False)
  

## Kendi tokenizer'ımız: UstaTokenizer

In [ ]:
from usta_tokenizer import UstaTokenizer

tokenizer = UstaTokenizer("tokenizer.json")

tokenizer.encode("states"), tokenizer.decode([4, 58])

In [ ]:
with open("text.txt", "r") as f:
  text = f.read()

text

In [ ]:
tokens = tokenizer.encode(text)
tokens

## Alt kelime (BPE) tokenizer eğitmek: SentencePiece

In [ ]:
#Tokenizer vocabı oluşturma.
import sentencepiece as spm

spm.SentencePieceTrainer.Train(
  input="text.txt",
  model_prefix="spm_tokenizer",
  vocab_size=64,
  model_type="bpe",
)

In [ ]:
spm_tokenizer = spm.SentencePieceProcessor(model_file="spm_tokenizer.model")

spm_ids = spm_tokenizer.Encode(text1)
spm_tokens = spm_tokenizer.Encode(text1, out_type=str)
spm_ids, spm_tokens

## Hugging Face `tokenizers` ile BPE

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

In [ ]:
hf_tokenizer = Tokenizer(BPE())

In [ ]:
hf_tokenizer.pre_tokenizer = Whitespace()

In [ ]:
trainer = BpeTrainer(vocab_size=64, special_tokens=["<unk>"])

hf_tokenizer.train(["text.txt"], trainer)

In [ ]:
hf_tokenizer.get_vocab_size(), hf_tokenizer.encode(text1).ids

In [ ]:
hf_tokenizer.save("hf_tokenizer.json")

In [ ]:
from transformers import PreTrainedTokenizerFast

fast_tokenizer = PreTrainedTokenizerFast(tokenizer_file="hf_tokenizer.json")

fast_tokenizer.encode(text1)